# 02 — Baseline CNN

Train the small CNN to validate the end-to-end training pipeline.  
For a full run use `python scripts/train_baseline.py` from `model/`; this notebook does a **short smoke run** (2 epochs).

**Prerequisite:** model package installed in the active kernel (see `model/README.md`).

In [1]:
from model_service.config import ModelServiceConfig
from model_service.preprocess.dataset_builder import build_pcam_datasets
from model_service.evaluation.plots import plot_history
from model_service.evaluation.metrics import calculate_and_save_metrics
from model_service.training.baseline import build_baseline_cnn, run_training


In [2]:
train_ds, val_ds, _, _ = build_pcam_datasets(
     download=True, use_efficientnet_preprocess=False
)
model = build_baseline_cnn()
model.summary(expand_nested=True)

2026-04-23 09:51:32.804159: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4 Pro
2026-04-23 09:51:32.804189: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2026-04-23 09:51:32.804191: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.00 GB
2026-04-23 09:51:32.804210: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-23 09:51:32.804218: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "baseline_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 94, 94, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 47, 47, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 45, 45, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 22, 22, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,505 (84.00 KB)

 Trainable params: 21,505 (84.00 KB)

 Non-trainable params: 0 (0.00 B)

In [3]:
# Absolute paths so saving works wherever Jupyter was launched
config = ModelServiceConfig()
models_dir   = config.paths.artifacts_models
metrics_dir  = config.paths.artifacts_metrics
figures_dir  = config.paths.artifacts_figures

for d in [models_dir, metrics_dir, figures_dir]:
    d.mkdir(parents=True, exist_ok=True)


hist = run_training(model, train_ds, val_ds)
metrics = calculate_and_save_metrics(hist)
plot_history(hist, figures_dir / "baseline_nb_history.png")
print("Training complete. Best checkpoint:", models_dir / "baseline_nb.keras")
print("Training complete. Metrics:", metrics)

Epoch 1/2


2026-04-23 09:51:37.204366: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


8192/8192 ━━━━━━━━━━━━━━━━━━━━ 166s 20ms/step - accuracy: 0.7516 - auc: 0.8263 - loss: 0.5091 - precision: 0.7529 - recall: 0.7478 - val_accuracy: 0.7968 - val_auc: 0.8948 - val_loss: 0.4294 - val_precision: 0.8442 - val_recall: 0.7274
Epoch 2/2
8192/8192 ━━━━━━━━━━━━━━━━━━━━ 167s 20ms/step - accuracy: 0.8258 - auc: 0.9027 - loss: 0.3965 - precision: 0.8278 - recall: 0.8218 - val_accuracy: 0.8275 - val_auc: 0.9153 - val_loss: 0.3735 - val_precision: 0.8381 - val_recall: 0.8113


NameError: name 'timestamp' is not defined